In [ ]:
# 실습에 필요한 패키지 설치 (최초 1회 실행)
!pip install -q openai python-dotenv pandas


# 2. 데이터 합성 (Synthesis)

> **📌 이 노트북의 목표**
>
> LLM(거대 언어 모델)을 활용하여 **완전히 새로운 학습 데이터를 생성**하고,
> 생성된 데이터의 품질을 **LLM이 직접 평가**하는 파이프라인을 구현한다.
>
> 이번 실습에서는 **'영화 추천'** 태스크를 위한 합성 데이터를 만들어 본다.

**이 노트북에서 다루는 핵심 개념 4가지:**
1. **LLM 호출 함수화** — 재사용 가능한 API 호출 패턴
2. **프롬프팅 기법 비교** — Zero-shot, Few-shot, CoT의 차이를 체험
3. **구조화된 합성 데이터 생성** — JSON 스키마로 일관된 데이터 생성
4. **LLM as Judge 평가** — 생성 데이터의 품질을 LLM이 자동 평가


## 2-1. 데이터 생성을 위한 Prompt Engineering

### 2-1-1. 합성 데이터(Synthetic Data)란?

실제 수집한 데이터가 아닌, **LLM을 통해 인공적으로 만들어낸 데이터**를 의미한다.

> **💡 합성(Synthesis) vs 증강(Augmentation)의 차이**
>
> | 구분 | 데이터 합성 | 데이터 증강 |
> |------|:---:|:---:|
> | 방식 | **완전히 새로운 데이터**를 생성 | 기존 데이터를 **변형**하여 양을 늘림 |
> | 비유 | 새로운 소설을 집필 | 기존 소설의 문장 순서를 바꾸거나 단어를 교체 |
> | 예시 | "공포 영화 추천해줘" → 새 질문-답변 쌍 생성 | 이미지 회전, 밝기 조절, 동의어 교체 |
> | 필요 시점 | 학습 데이터가 **없거나 부족**할 때 | 기존 데이터가 있지만 **양이 부족**할 때 |
>
> 3-1 챕터에서 배운 데이터 증강(RandomCrop, Flip 등)은 기존 이미지를 변형한 것이었다.
> 여기서는 **없던 데이터를 LLM으로 새로 만드는** 합성 방식을 다룬다.

합성 데이터는 다음과 같은 경우에 유용하다:
- 학습 데이터가 **부족**할 때
- 실제 데이터를 수집하기 어려운 **민감한 정보**(개인정보, 의료정보 등)를 다뤄야 할 때
- 특정 도메인의 **다양한 시나리오**를 빠르게 만들어야 할 때

### 2-1-2. 프롬프트의 기본 구조

양질의 합성 데이터를 만들려면 LLM에게 **명확한 가이드라인**을 제시해야 한다.

1. **역할(Role)**: AI에게 전문성과 톤앤매너를 설정
   - 예: "당신은 세계적인 영화 평론가입니다."
2. **목표(Task)**: 수행해야 할 작업을 구체적으로 지시
   - 예: "사용자의 취향에 맞는 영화를 추천해 주세요."
3. **조건(Constraints)**: 형식, 스타일, 분량 등 제약 조건
   - 예: "영화는 반드시 한 편만 추천하고, 이유는 세 문장 이내로."

### 2-1-3. 합성 데이터 프롬프트의 핵심

양질의 합성 데이터를 만들기 위해서는 **다양성**과 **일관성**을 모두 확보해야 한다.

**1. 다양성 확보 (temperature, top_p)**

모델이 매번 새롭고 다채로운 데이터를 생성하도록 무작위성을 조절한다.

| 파라미터 | 역할 | 낮은 값 | 높은 값 |
|---------|------|--------|--------|
| `temperature` | 확률 분포의 모양 조절 | 일관적·결정적 답변 (0에 가까울수록) | 창의적·다양한 답변 (1에 가까울수록) |
| `top_p` | 후보 단어의 범위 조절 | 소수의 안전한 단어만 후보 | 더 많은 단어를 후보에 포함 |



> **💡 temperature vs top_p**
>
> `temperature`는 확률 분포의 **뾰족함/완만함**을 조절하고,
> `top_p`는 누적 확률이 일정 값에 도달할 때까지의 **후보군 범위**를 조절한다.
> 보통 둘 중 하나만 조절하고, 나머지는 기본값(1.0)으로 두는 것이 권장된다.


> **💡 [참고] 최신 모델에서의 temperature / top_p 변화**
>
> 이 실습에서 사용하는 Solar 같은 생성 모델에서는 temperature/top_p가 정상 동작한다.
> 하지만 OpenAI의 최신 추론 모델(o3, GPT-5 계열)에서는 이 파라미터가 **아예 비활성화**되었다.
>
> - **이유**: 추론 모델은 내부적으로 여러 경로를 탐색 → 검증 → 선택하는 복잡한 파이프라인을 사용한다. 외부에서 temperature를 조절하면 이 내부 보정이 깨지기 때문이다.
> - **대체**: `reasoning_effort` (none ~ xhigh) 파라미터로 추론 깊이를 조절하는 방식으로 전환되었다.
> - **참고**: [GPT-5 Prompting Guide](https://developers.openai.com/cookbook/examples/gpt-5/gpt-5_prompting_guide)

**2. 일관성 확보 (response_format)**

생성된 데이터가 항상 같은 구조(예: `{"movie_name": ..., "year": ...}`)를 갖도록
JSON 스키마를 지정하여 **출력 형식을 강제**한다.
이는 후속 처리(DB 저장, 모델 학습 등)를 자동화하는 데 필수적이다.


### 2-1-4. 프롬프팅 기법 비교

동일한 작업이라도 **프롬프트를 어떻게 설계하느냐**에 따라 LLM의 응답 품질이 달라진다.
대표적인 세 가지 기법을 알아보자.

| 기법 | 설명 | 장점 | 단점 |
|------|------|------|------|
| **Zero-shot** | 예시 없이 직접 지시 | 간단, 토큰 절약 | 복잡한 작업에 한계 |
| **Few-shot** | 2~3개의 입출력 예시를 함께 제공 | 원하는 패턴/형식 유도 가능 | 토큰 소비 증가 |
| **Chain-of-Thought (CoT)** | "단계별로 생각해보세요" 지시 추가 | 논리적 정확성 향상 | 응답 길이 증가 |

아래 코드에서 세 가지 기법의 실제 출력 차이를 직접 확인해 보자.

**Chain-of-Thought (CoT)란?**
- LLM이 복잡한 문제를 해결할 때 **중간 추론 과정을 명시적으로 생성**하도록 유도하는 기법
- "단계별로 생각해보세요(Let's think step by step)"와 같은 지시를 추가
- 단순 질문보다 **복잡한 추론, 분석, 의사결정**이 필요할 때 효과적


## 2-2. 실습 코드

### 2-2-1. 환경 설정

2_1_Setting에서 설정한 것과 동일한 방식으로 API 키를 로드한다.


In [ ]:
import json
import pandas as pd
from pprint import pprint
from dotenv import load_dotenv
from os import getenv
from openai import OpenAI

# .env 파일에서 API 키 로드
load_dotenv()
UPSTAGE_API_KEY = getenv('UPSTAGE_API_KEY')

if UPSTAGE_API_KEY:
    print('API 키 로드 성공!')
else:
    print('ERROR: .env 파일에 UPSTAGE_API_KEY가 설정되지 않았습니다.')


### 2-2-2. LLM 호출 함수 구현

API 호출을 매번 직접 작성하면 코드가 반복된다.
**재사용 가능한 함수**로 만들어 두면 이후 모든 호출에서 편리하게 사용할 수 있다.

> **💡 왜 함수로 만드는가?**
>
> 이 노트북에서만 해도 LLM을 **데이터 생성, 프롬프팅 비교, 품질 평가** 등 여러 번 호출한다.
> 매번 `client.chat.completions.create(...)`를 작성하는 대신,
> `chat_completion(prompt)` 한 줄로 호출할 수 있도록 함수화한다.


In [ ]:
# ========== Upstage API 클라이언트 생성 ==========
client = OpenAI(
    api_key=UPSTAGE_API_KEY,
    base_url='https://api.upstage.ai/v1',
)

# ========== 재사용 가능한 LLM 호출 함수 ==========
# 왜 함수로 만드는가?
# → 이 실습에서 LLM을 여러 번 호출한다 (데이터 생성, 평가 등).
#   매번 client.chat.completions.create(...)를 작성하면 코드가 반복된다.
#   함수로 만들면 prompt만 바꿔가며 재사용할 수 있다.
def chat_completion(
    prompt: str,
    system_prompt: str = None,
    model: str = 'solar-pro3',
    temperature: float = 1.0,
    response_format: dict = None,
) -> str:
    """
    LLM을 호출하여 응답 텍스트를 반환하는 함수.

    Args:
        prompt: 사용자 메시지
        system_prompt: 시스템 메시지 (선택)
        model: 사용할 모델명
        temperature: 응답 다양성 (0=일관적, 1=창의적)
        response_format: JSON 스키마 (선택)

    Returns:
        LLM의 응답 텍스트 (문자열)
    """
    messages = []
    if system_prompt:
        messages.append({'role': 'system', 'content': system_prompt})
    messages.append({'role': 'user', 'content': prompt})

    kwargs = {
        'model': model,
        'messages': messages,
        'temperature': temperature,
    }
    if response_format:
        kwargs['response_format'] = response_format

    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message.content


# 테스트
test = chat_completion('안녕? 간단한 인사말을 해줘.')
print('테스트 응답:', test)


### 2-2-3. 프롬프팅 기법 비교 (Zero-shot / Few-shot / CoT)

동일한 작업(영화 추천)에 대해 세 가지 기법을 적용하고, 결과 차이를 직접 확인한다.

> **💡 관찰 포인트**
>
> 세 기법의 출력을 비교할 때 다음을 주목하자:
> - **형식의 일관성**: Few-shot이 Zero-shot보다 형식을 더 잘 맞추는가?
> - **추론의 깊이**: CoT가 단순 추천을 넘어 논리적 분석을 보여주는가?
> - **토큰 소비**: 프롬프트가 길수록 응답에 어떤 영향이 있는가?


In [ ]:
# 공통 시스템 프롬프트 (세 기법 모두 동일한 역할 부여)
SYSTEM_PROMPT = '''당신은 세상의 모든 영화를 꿰뚫고 있는 영화 전문가 \'시네마스터\'입니다.
사용자의 요청에 맞춰 영화를 추천하는 역할을 맡고 있습니다.
추천할 때는 반드시 영화 제목, 개봉 연도, 그리고 추천 이유를 포함해야 합니다.'''

user_query = '공포 영화를 추천해줘'

In [ ]:
# ===== 1. Zero-shot: 예시 없이 직접 지시 =====
zero_shot_prompt = user_query  # 아무 예시도 없이 그냥 질문

print('=' * 50)
print('1. Zero-shot 결과:')
print('=' * 50)
print(chat_completion(zero_shot_prompt, SYSTEM_PROMPT))


In [ ]:
# ===== 2. Few-shot: 입출력 예시 2개 제공 =====
# 왜 예시를 주는가?
# → LLM에게 "이런 형식으로 대답해"라는 패턴을 보여주는 것
#   예시가 있으면 형식과 톤앤매너를 더 정확히 맞출 수 있다
few_shot_prompt = f'''다음은 영화 추천 예시입니다:

질문: 로맨스 영화를 추천해줘
답변: 영화 제목: 노트북 (The Notebook)
개봉 연도: 2004년
추천 이유: 시대를 초월한 순수한 사랑 이야기로, 감동적인 클래식 로맨스 영화입니다.

질문: 코미디 영화를 추천해줘
답변: 영화 제목: 행오버 (The Hangover)
개봉 연도: 2009년
추천 이유: 예측 불가능한 전개와 유쾌한 유머가 가득한 코미디 영화입니다.

질문: {user_query}
답변:'''

print('\n' + '=' * 50)
print('2. Few-shot 결과:')
print('=' * 50)
print(chat_completion(few_shot_prompt, SYSTEM_PROMPT))

In [ ]:
# ===== 3. Chain-of-Thought: 단계별 추론 유도 =====
# 왜 단계를 나누는가?
# → 복잡한 작업에서 LLM이 "생각하는 과정"을 거치면 더 정확한 결과를 낸다
cot_prompt = f'''{user_query}

영화를 추천하기 전에 다음 단계를 따라 생각해주세요:
1단계: 공포 장르의 핵심 요소가 무엇인지 정의합니다
2단계: 이 요소들을 잘 갖춘 대표적인 공포 영화들을 떠올립니다
3단계: 그 중 가장 추천할 만한 영화 1개를 선택하고 이유를 설명합니다

위 단계를 따라 추론 과정을 보여주고, 최종 추천을 해주세요.'''

print('\n' + '=' * 50)
print('3. Chain-of-Thought 결과:')
print('=' * 50)
print(chat_completion(cot_prompt, SYSTEM_PROMPT))

### 2-2-4. JSON 파싱 방법 2가지

LLM의 응답을 구조화된 데이터로 받는 방법은 두 가지가 있다.

| 방법 | 설명 | 장점 | 단점 |
|------|------|------|------|
| `response_format` (API 레벨) | API 파라미터로 JSON 스키마 강제 | 파싱 실패 없음, 정확함 | 일부 API만 지원 |
| **프롬프트에 JSON 형식 지정** + 수동 파싱 | 프롬프트에 "JSON으로 답해"라고 지시 | 어떤 API든 사용 가능 | 파싱 실패 가능성 있음 |

이 노트북에서는 `response_format`(방법 1)을 주로 사용하되,
범용적으로 사용할 수 있는 `json_parsing()` 유틸리티 함수(방법 2)도 함께 구현해 둔다.


In [ ]:
# ========== JSON 파싱 유틸리티 함수 ==========
# LLM이 ```json ... ``` 형태로 응답할 때, 그 안의 JSON만 추출하는 함수
# 왜 필요한가?
# → response_format을 지원하지 않는 API도 있고,
#   프롬프트로 JSON을 요청하면 앞뒤에 설명 텍스트가 붙는 경우가 많다.
#   이 함수는 그 중에서 JSON 부분만 깔끔하게 추출한다.
def json_parsing(output_text: str) -> dict:
    """
    LLM 응답에서 JSON 부분을 추출하여 딕셔너리로 변환.

    Args:
        output_text: LLM의 응답 텍스트

    Returns:
        파싱된 딕셔너리, 실패 시 None
    """
    try:
        # ```json ... ``` 블록이 있으면 그 안의 내용만 추출
        if '```json' in output_text:
            output_text = output_text[output_text.index('```json') + len('```json'):].strip()
            output_text = output_text[:output_text.index('```')].strip()
        return json.loads(output_text)
    except (json.JSONDecodeError, ValueError) as e:
        print(f'JSON 파싱 오류: {e}')
        return None

print('json_parsing 함수 정의 완료')


### 2-2-5. 구조화된 합성 데이터 생성

프롬프트 엔지니어링의 3요소(역할, 목표, 조건)를 적용하여 영화 추천 데이터를 생성한다.
여기서는 `response_format`(API 레벨 강제)을 사용한다.

> **💡 의도적으로 규칙을 지키기 어려운 데이터도 함께 생성한다**
>
> 모든 데이터가 완벽하면 이후 평가 단계에서 점수 변별력이 없어진다.
> 아래에서 **RULE이 없는 "일반 생성"과 RULE이 있는 "규칙 생성"**을 모두 만들어서
> 평가 시 점수 차이가 나타나도록 구성한다.


In [ ]:
# ========== 1. 프롬프트 설계 ==========

# 시스템 프롬프트: 역할(Role) + 목표(Task)
GEN_SYSTEM_PROMPT = '''당신은 세상의 모든 영화를 꿰뚫고 있는 영화 전문가 \'시네마스터\'입니다.
사용자의 요청에 맞춰 영화를 추천하는 역할을 맡고 있습니다. 영화는 반드시 하나만 추천합니다.'''

# 추가 규칙: 조건(Constraints) — 말투/스타일 통제
RULE = '''친구가 소개해주는 듯 부드럽고 친근한 말투로 답변합니다.
특히, recommended_reason 항목에서는 친구가 엄청 호들갑 떨듯이 설명해 주세요.'''

# ========== 2. JSON 응답 형식 (API 레벨 강제) ==========
response_format = {
    'type': 'json_schema',
    'json_schema': {
        'name': '영화 추천',
        'strict': True,
        'schema': {
            'type': 'object',
            'properties': {
                'movie_name': {'type': 'string'},
                'year': {'type': 'integer'},
                'reason': {'type': 'string'},
                'description': {'type': 'string', 'description': '영화에 대한 설명'},
                'recommended_reason': {'type': 'string', 'description': '추천 추가 이유'},
            },
            'required': ['movie_name', 'year', 'reason', 'description', 'recommended_reason'],
        },
    },
}

# ========== 3. 합성 데이터 생성 ==========
# 점수 다양성을 확보하기 위해 두 가지 조건으로 생성한다:
# - RULE 적용 (친근한 말투 규칙을 따르는 데이터)
# - RULE 미적용 (규칙 없이 기본 톤으로 생성한 데이터)
# → 이후 평가에서 RULE 준수 여부를 기준으로 평가하면 점수 차이가 발생한다.

generation_configs = [
    {'genre': '공포', 'use_rule': True,  'label': '공포 (RULE 적용)'},
    {'genre': 'SF',   'use_rule': True,  'label': 'SF (RULE 적용)'},
    {'genre': '액션', 'use_rule': False, 'label': '액션 (RULE 미적용)'},
    {'genre': '로맨스', 'use_rule': False, 'label': '로맨스 (RULE 미적용)'},
]

synthetic_data = []

for config in generation_configs:
    genre = config['genre']
    label = config['label']
    # RULE 적용 여부에 따라 시스템 프롬프트를 다르게 구성
    if config['use_rule']:
        sys_prompt = GEN_SYSTEM_PROMPT + '\n' + RULE
    else:
        sys_prompt = GEN_SYSTEM_PROMPT  # RULE 없이 기본 톤

    print(f'\n{label} 생성 중...')
    output = chat_completion(
        prompt=f'{genre} 영화를 추천해줘',
        system_prompt=sys_prompt,
        temperature=0.5,
        response_format=response_format,
    )
    data = json.loads(output)
    data['_rule_applied'] = config['use_rule']  # 메타 정보: 평가 시 참고용
    data['_genre'] = genre
    synthetic_data.append(data)
    pprint(data)

print(f'\n생성 완료: 총 {len(synthetic_data)}건 (RULE 적용 {sum(c["use_rule"] for c in generation_configs)}건, 미적용 {sum(not c["use_rule"] for c in generation_configs)}건)')


## 2-3. 생성 데이터 평가 (LLM as a Judge)

### 2-3-1. LLM as a Judge란?

사람이 수많은 합성 데이터의 품질을 하나하나 검수하는 것은 비효율적이다.
**LLM as a Judge**는 이 검수 과정을 자동화하기 위해 **다른 LLM을 '평가자'로 활용**하는 기법이다.

> **💡 핵심 장점**
>
> 단순히 "좋다/나쁘다" 판정을 넘어,
> **"왜 그렇게 평가했는지" 이유까지 생성**하여 데이터의 어떤 부분을 개선해야 할지
> 구체적인 피드백을 얻을 수 있다.

### 2-3-2. 평가 설계의 핵심 3가지

1. **평가 기준 설정**
   - 평가 목적을 명확히 해야 한다
   - 이번 실습: 영화 정보의 사실 여부가 아니라, 우리가 지시한 **RULE(친근한 말투, 호들갑 떠는 설명)을 얼마나 잘 이행했는지**를 평가

2. **일관성 확보 (temperature=0)**
   - 평가자 LLM은 창의적일 필요가 없다
   - 동일한 입력에 대해 항상 **일관되고 객관적인 평가**를 내려야 한다
   - 따라서 `temperature=0`으로 설정하여 **결정적(deterministic)** 응답을 유도한다

3. **체계적인 평가 프롬프트**
   - 평가자에게 **[원래 지시사항], [생성 결과], [평가 기준]**을 모두 명시적으로 전달
   - 출력도 `score`(점수)와 `comment`(이유)를 포함하는 **JSON 형식**으로 강제


### 2-3-3. 평가 실습 코드

아래에서 평가 함수를 정의하고, 먼저 단일 데이터에 대해 테스트한 뒤,
모든 합성 데이터에 대해 다중 기준으로 평가하는 과정까지 진행한다.


In [ ]:
# ========== 1. 평가자 시스템 프롬프트 ==========
JUDGE_SYSTEM_PROMPT = '''당신의 역할은 모델 답변 자동 평가자입니다.

1. 입력 형식
    - 입력 프롬프트: [instruction]
    - 모델 답변: [output]
    - 평가 기준: [criteria]

2. 작업 지시
    - [instruction]에 따른 모델 결과물인 [output]을 평가합니다.
    - [output]은 [criteria]를 충족하는지 평가합니다.

3. 채점 원칙 (1–5점, 정수만)
    - 5점 (탁월): 기준을 완전히 충족.
    - 4점 (우수): 대체로 충족. 사소한 흠만 있음.
    - 3점 (보통): 핵심은 맞지만 약점 존재.
    - 2점 (미흡): 중요한 요구를 놓침.
    - 1점 (부적합): 전반적으로 요청과 어긋남.

4. 출력 형식 (엄격 준수)
    - "score"는 1–5점 정수.
    - "comment"는 한국어 1–3문장.
    - JSON 형식의 response_format을 준수.'''

# ========== 2. 평가 JSON 응답 형식 ==========
judge_response_format = {
    'type': 'json_schema',
    'json_schema': {
        'name': '평가 결과',
        'strict': True,
        'schema': {
            'type': 'object',
            'properties': {
                'score': {'type': 'integer'},
                'comment': {'type': 'string', 'description': '평가 이유'},
            },
            'required': ['score', 'comment'],
        },
    },
}

# ========== 3. 평가 함수 ==========
# 여러 데이터 × 여러 기준으로 반복 평가해야 하므로 함수로 만든다.
def evaluate_with_llm(instruction: str, output: str, criteria: str) -> dict:
    """
    LLM as Judge: 생성 결과를 평가하여 점수와 코멘트를 반환.

    Args:
        instruction: 원본 지시문 (데이터 생성 시 사용한 프롬프트)
        output: 평가할 생성 결과
        criteria: 평가 기준 문자열

    Returns:
        {'score': 1~5, 'comment': '평가 이유'}
    """
    user_prompt = f'''입력 프롬프트: {instruction}
모델 답변: {output}
평가 기준: {criteria}'''

    result_text = chat_completion(
        prompt=user_prompt,
        system_prompt=JUDGE_SYSTEM_PROMPT,
        temperature=0,  # 평가 일관성을 위해 반드시 0
        response_format=judge_response_format,
    )
    return json.loads(result_text)


# ========== 4. 단일 평가 테스트 ==========
test_data = synthetic_data[0]
print('평가 대상:')
pprint(test_data)

result = evaluate_with_llm(
    instruction=GEN_SYSTEM_PROMPT,
    output=json.dumps(test_data, ensure_ascii=False),
    criteria='요청의 충실도: 요청된 장르에 맞는 영화를 추천했는지, 필수 정보가 모두 포함되었는지',
)
print(f'\n점수: {result["score"]} / 5')
print(f'평가: {result["comment"]}')


### 2-3-4. 다중 기준 평가 + 결과 정리

실제 품질 평가에서는 **하나의 기준이 아닌 여러 기준**으로 평가해야 한다.
모든 합성 데이터에 대해 여러 기준으로 평가하고, 결과를 표로 정리해 보자.

> **💡 왜 RULE 적용/미적용 데이터를 섞었는가?**
>
> 모든 데이터가 동일한 조건으로 생성되면 평가 점수가 전부 비슷하게 나와서
> **점수의 변별력**을 확인하기 어렵다.
> RULE(친근한 말투 규칙)이 적용된 데이터와 그렇지 않은 데이터를 함께 평가하면,
> "말투 규칙 준수" 기준에서 **점수 차이가 나타나는 것**을 관찰할 수 있다.
> 이것이 LLM as Judge의 **변별력을 검증**하는 방법이다.


In [ ]:
# ========== 다중 평가 기준 정의 ==========
# 점수 다양성을 위해 "쉬운 기준"과 "까다로운 기준"을 함께 배치한다.
# - 기준 1~2: 대부분의 데이터가 충족하기 쉬운 기본 기준
# - 기준 3: RULE을 적용한 데이터만 높은 점수를 받을 수 있는 까다로운 기준
criteria_list = [
    '요청의 충실도: 요청된 장르에 맞는 영화를 추천했는지, 필수 정보(영화명, 연도, 이유)가 모두 포함되었는지 평가',
    '추천 이유의 구체성: 추천 이유가 해당 장르의 특성과 연결되어 구체적이고 설득력 있는지, 단순히 "재미있다" 수준의 피상적 설명은 3점 이하로 평가',
    '말투 규칙 준수: 친구가 소개해주는 듯 부드럽고 친근한 말투인지, 특히 추천 이유에서 호들갑 떠는 듯한 열정적 표현이 사용되었는지 평가. 격식체나 딱딱한 문어체는 2점 이하로 평가',
]

# ========== 모든 데이터 × 모든 기준 평가 ==========
# 평가 시 instruction에 RULE을 포함시킨다.
# → RULE 없이 생성된 데이터는 "말투 규칙 준수" 기준에서 낮은 점수를 받게 된다.
evaluation_instruction = GEN_SYSTEM_PROMPT + '\n' + RULE

evaluation_results = []
print('모든 합성 데이터 평가 시작...\n')

for i, data in enumerate(synthetic_data):
    rule_tag = '✅ RULE 적용' if data.get('_rule_applied') else '❌ RULE 미적용'
    print(f'[{i+1}] {data.get("movie_name", "Unknown")} ({rule_tag}) 평가 중...')
    data_output = json.dumps(data, ensure_ascii=False)

    scores = []
    comments = []
    for criteria in criteria_list:
        result = evaluate_with_llm(
            instruction=evaluation_instruction,
            output=data_output,
            criteria=criteria,
        )
        scores.append(result.get('score', 0))
        comments.append(result.get('comment', ''))

    avg_score = sum(scores) / len(scores) if scores else 0
    evaluation_results.append({
        'data': data,
        'scores': scores,
        'comments': comments,
        'avg': avg_score,
    })
    print(f'   점수: {scores}, 평균: {avg_score:.2f}')

# ========== 결과를 DataFrame으로 정리 ==========
criteria_labels = ['충실도', '구체성', '말투규칙']
rows = []
for r in evaluation_results:
    row = {
        'movie_name': r['data'].get('movie_name', ''),
        'genre': r['data'].get('_genre', ''),
        'RULE': '적용' if r['data'].get('_rule_applied') else '미적용',
    }
    for j, label in enumerate(criteria_labels):
        row[label] = r['scores'][j]
    row['평균'] = r['avg']
    rows.append(row)

df = pd.DataFrame(rows)
print('\n=== 평가 결과 요약 ===')
print(df.to_string(index=False))

# RULE 적용 vs 미적용 그룹별 평균 비교
print('\n=== RULE 적용 여부별 평균 점수 ===')
print(df.groupby('RULE')['평균'].mean().to_string())

print('\n→ RULE이 적용된 데이터는 "말투규칙" 기준에서 높은 점수를,')
print('  RULE이 미적용된 데이터는 낮은 점수를 받는 것을 확인할 수 있다.')
print('  이것이 LLM as Judge의 변별력이다.')


## [참고] 데이터 증강 (Data Augmentation)

이번 실습에서는 LLM을 활용한 **데이터 합성(Synthesis)**, 즉 완전히 새로운 데이터를 생성하는 방법에 초점을 맞추었다.

관련 기법으로 **데이터 증강(Data Augmentation)**이 있다:
- 기존 데이터를 기반으로 **변형**(이미지 회전, 텍스트 동의어 교체)을 가해 데이터 양을 늘리는 방식
- 모델의 강건성(Robustness)을 높이고 과적합을 방지하는 데 도움

> **💡 한마디로 정리하면**
>
> - **합성**: 무(無)에서 유(有)를 창조 (새 질문-답변 쌍 생성)
> - **증강**: 유(有)에서 또 다른 유(有)를 생성 (기존 데이터 변형)
>
> 3-1 챕터의 `RandomResizedCrop`, `RandomHorizontalFlip` 등이 데이터 증강에 해당한다.
